# 04 - Join e anti-leakage

I notebook 02 e 03 hanno prodotto due tabelle indipendenti: le osservazioni ISD
(cio' che e' successo) e i punti di previsione GFS (cio' che il modello aveva
detto). Nessuna delle due, da sola, permette di valutare una previsione: serve
unirle sullo stesso istante e sulla stessa stazione.

Questo e' il notebook concettualmente piu' importante del percorso. Il join
temporale e' anche il punto in cui un dataset di training si rovina in
silenzio: se una feature contiene informazione che al momento della
previsione non era ancora disponibile, il modello sembra ottimo in backtest e
fallisce in produzione. Questo fenomeno si chiama **leakage** (fuga di
informazione dal futuro), ed e' la regola che il documento
`docs/02-historical-forecast-observation-protocol.md` mette al primo posto
fra le regole anti-leakage: *nessuna feature puo' usare informazione non
disponibile entro `publication_time_utc`*.

## Passo 1 - Prerequisiti e join temporale

Il notebook eredita due artefatti:

- `02_observations.parquet`: osservazioni ISD, una riga per stazione e istante
  di misura, con il flag di qualita' `qa_flag`.
- `03_forecast_points.parquet`: previsioni GFS gia' interpolate sul punto
  stazione, con `run_time_utc`, `publication_time_utc`, `lead_hours` e
  `valid_time_utc`.

Le osservazioni ISD non arrivano ogni ora esatta: si arrotonda l'istante di
misura all'ora piena (`floor("h")`) e si media, cosi' da avere una riga per
stazione e ora, confrontabile con l'uscita oraria del modello. Le righe con
flag di qualita' sospetti (`2`, `3`, `6`, `7`, secondo la codifica ISD) vengono
escluse prima della media, non dopo: un valore scartato non deve contaminare
la media di quelli buoni.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
import common
import pandas as pd

common.richiede("02_observations.parquet", "03_forecast_points.parquet")
oss = pd.read_parquet(common.data_path("02_observations.parquet"))
fc = pd.read_parquet(common.data_path("03_forecast_points.parquet"))

# Il forecast e' orario: si arrotonda l'osservazione all'ora piena e si
# uniscono le due tabelle su (stazione, istante di validita').
oss["ora_utc"] = oss["valid_time_utc"].dt.floor("h")
oss_oraria = (oss[~oss["qa_flag"].isin(list("2367"))]
              .groupby(["station_id", "ora_utc"], as_index=False)["t2m_c"].mean()
              .rename(columns={"t2m_c": "t2m_c_osservato", "ora_utc": "valid_time_utc"}))

ds_reale = fc.merge(oss_oraria, on=["station_id", "valid_time_utc"], how="inner")
if len(ds_reale) > 0:
    ds_reale["errore"] = ds_reale["t2m_c_forecast"] - ds_reale["t2m_c_osservato"]
print(f"Coppie forecast-osservazione: {len(ds_reale)} su {len(fc)} forecast")
if len(ds_reale) > 0:
    print(ds_reale[["station_id", "lead_hours", "t2m_c_forecast", "t2m_c_osservato", "errore"]].head(10).to_string(index=False))
else:
    print(f"Intervallo osservazioni:  {oss['valid_time_utc'].min()} .. {oss['valid_time_utc'].max()}")
    print(f"Intervallo previsione:    {fc['valid_time_utc'].min()} .. {fc['valid_time_utc'].max()}")

### Perche' il join reale e' vuoto, e cosa si fa in questo caso

Il risultato sopra e' **zero righe**, e non e' un bug del codice. Le stazioni
coincidono (lo stesso set di 5 `station_id` compare in entrambe le tabelle),
ma gli istanti no: `02_observations.parquet` copre il 2024, mentre
`03_forecast_points.parquet` e' un run scaricato oggi dal notebook 03, quindi
si riferisce ad agosto 2026. Una previsione scaricata oggi non puo' essere
verificata contro osservazioni di due anni fa: non esiste nessun istante in
comune su cui fare il join.

Questo e' esso stesso un insegnamento sull'allineamento temporale: una
pipeline vera unisce previsioni e osservazioni **dello stesso periodo**, e i
notebook didattici, scaricando i due input in momenti diversi del calendario,
hanno costruito senza volerlo un caso limite che lo dimostra a costo zero.

Per non fermarsi qui e poter comunque mostrare leakage e split sui dati reali,
si costruisce ora un **dataset sintetico**, dichiarato come tale, generato con
seed fisso per riproducibilita'. Non sostituisce il dato reale ne' lo
maschera: viene tenuto distinto (`ds_sintetico`) e usato solo perche' il join
vero, in questo campione, non produce righe sufficienti a dimostrare nulla.
Struttura: si simulano 30 giorni di temperatura oraria autocorrelata (una
passeggiata casuale attorno a un ciclo diurno) per le stesse 5 stazioni delle
osservazioni reali, con un run di previsione per ciascuna ora e un errore di
previsione casuale ma plausibile.

In [ ]:
import numpy as np

# =====================================================================
# DATASET SINTETICO - dichiarato come tale, NON e' un'osservazione reale.
# Serve solo perche' il join reale sopra produce 0 righe (osservazioni 2024,
# previsione 2026: nessun istante in comune). Generato con seed fisso.
# =====================================================================
rng_sint = np.random.default_rng(20260829)

stazioni_sintetiche = sorted(oss["station_id"].unique())
ore = pd.date_range("2024-06-01", periods=24 * 30, freq="h", tz="UTC")

righe_sintetiche = []
for stazione in stazioni_sintetiche:
    # Ciclo diurno + passeggiata casuale: una serie autocorrelata, non rumore bianco.
    ciclo_diurno = 15 + 8 * np.sin(2 * np.pi * (ore.hour - 6) / 24)
    deriva = rng_sint.normal(0, 0.3, size=len(ore)).cumsum() * 0.1
    t2m_osservato = ciclo_diurno + deriva

    # Il "forecast" e' l'osservazione vera piu' un errore di previsione plausibile,
    # via via meno preciso al crescere del lead: cosi' com'e' un vero GFS.
    for lead in [3, 6, 12, 24, 48]:
        rumore_previsione = rng_sint.normal(0, 0.4 + 0.05 * lead, size=len(ore))
        t2m_previsto = t2m_osservato + rumore_previsione
        run_time = ore - pd.Timedelta(hours=lead)
        righe_sintetiche.append(pd.DataFrame({
            "station_id": stazione,
            "run_time_utc": run_time,
            "publication_time_utc": run_time + pd.Timedelta(hours=4),
            "lead_hours": lead,
            "valid_time_utc": ore,
            "t2m_c_forecast": t2m_previsto,
            "t2m_c_osservato": t2m_osservato,
        }))

ds_sintetico = pd.concat(righe_sintetiche, ignore_index=True)
ds_sintetico["errore"] = ds_sintetico["t2m_c_forecast"] - ds_sintetico["t2m_c_osservato"]

# Da qui in avanti il notebook lavora su ds_sintetico: e' l'unico modo, con
# questo campione, di avere righe sufficienti per leakage e split.
ds = ds_sintetico
print(f"Dataset sintetico: {len(ds)} righe, {ds['station_id'].nunique()} stazioni, "
      f"lead {sorted(ds['lead_hours'].unique())}")
print(ds[["station_id", "lead_hours", "valid_time_utc", "t2m_c_forecast", "t2m_c_osservato", "errore"]].head(10).to_string(index=False))

## Passo 2 - Perche' gli intervalli sono half-open

Per una temperatura istantanea come `t2m_c` basta un singolo istante:
`valid_time_utc` identifica il momento e non serve altro. Ma per una quantita'
**accumulata**, come la pioggia caduta in un'ora, il target e' un intervallo
`[inizio, fine)`, con l'estremo destro escluso.

La ragione e' evitare di contare due volte il confine: se l'intervallo fosse
chiuso su entrambi i lati (`[inizio, fine]`), l'istante `fine` apparterrebbe
sia all'intervallo corrente sia a quello successivo che inizia li'. Il
documento `docs/02-historical-forecast-observation-protocol.md` lo impone
esplicitamente (regola 3 delle regole anti-leakage): un disallineamento anche
di una sola ora fra l'intervallo previsto e quello osservato produce un
errore che non e' del modello meteorologico, ma del join che ha costruito il
dataset. In questo notebook il target e' istantaneo (temperatura), quindi la
regola non si applica direttamente ai dati che seguono, ma va ricordata prima
di estendere il metodo alla precipitazione.

## Passo 3 - Il leakage, dimostrato facendolo

> **ATTENZIONE: la cella che segue contiene codice deliberatamente
> sbagliato.** Serve unicamente a mostrare come si manifesta il leakage nei
> numeri (un miglioramento delle metriche che sembra un successo e invece e'
> una bugia). Il codice **non va copiato ne' riutilizzato**: la feature che
> costruisce e' vietata dalla regola 1 del documento 02 e la variabile e'
> nominata `feature_SBAGLIATA_non_usare` apposta, per non poter essere presa
> per una feature legittima per errore.

In [ ]:
# =====================================================================
# ATTENZIONE: CODICE DELIBERATAMENTE SBAGLIATO.
# Serve solo a mostrare come si manifesta il leakage. NON COPIARLO.
# =====================================================================
sbagliato = ds.copy()

# L'ERRORE: si usa come feature l'osservazione dell'ora successiva, che al
# momento della previsione NON ERA ANCORA ACCADUTA.
sbagliato = sbagliato.sort_values(["station_id", "valid_time_utc"])
sbagliato["feature_SBAGLIATA_non_usare"] = (
    sbagliato.groupby("station_id")["t2m_c_osservato"].shift(-1)
)
sbagliato = sbagliato.dropna(subset=["feature_SBAGLIATA_non_usare"])

# Una "correzione" che sfrutta la feature proibita.
previsione_barata = (sbagliato["t2m_c_forecast"] + sbagliato["feature_SBAGLIATA_non_usare"]) / 2
mae_barato = float(np.abs(previsione_barata - sbagliato["t2m_c_osservato"]).mean())
mae_onesto = float(np.abs(sbagliato["t2m_c_forecast"] - sbagliato["t2m_c_osservato"]).mean())

print(f"MAE del GFS grezzo:            {mae_onesto:.2f} C")
print(f"MAE del modello 'migliorato':  {mae_barato:.2f} C   <-- sembra fantastico")
print("\nE' una bugia: quella feature non esisteva al momento della previsione.")

## Passo 4 - Il sintomo, e il controllo automatico

Il miglioramento mostrato sopra e' vistoso e arriva senza il minimo sforzo di
modellazione: una semplice media con un valore futuro dimezza l'errore. **Un
salto di qualita' improvviso e inspiegabile e' il sintomo tipico del
leakage**, non un successo da festeggiare. La regola violata e' la prima
regola anti-leakage del documento 02: nessuna feature puo' usare informazione
non disponibile entro `publication_time_utc`.

In produzione l'errore non si vede finche' non e' troppo tardi: in backtest
il modello brilla perche' il "futuro" e' gia' nella tabella; dal vivo,
all'istante della previsione, quell'osservazione futura semplicemente non
esiste ancora, e le prestazioni crollano al livello onesto (o sotto, se il
codice va in errore per un valore mancante).

Il controllo automatico e sistematico e': per ogni feature, chiedersi da
quale istante proviene il suo valore, e confrontarlo con
`publication_time_utc`. Se l'istante di origine e' successivo alla
pubblicazione, la feature e' inammissibile.

In [ ]:
# Il controllo che va eseguito su ogni feature.
istante_feature = sbagliato.groupby("station_id")["valid_time_utc"].shift(-1)
violazioni = (istante_feature > sbagliato["publication_time_utc"]).sum()
print(f"Feature che usano informazione successiva alla pubblicazione: {violazioni} su {len(sbagliato)}")
assert violazioni > 0, "l'esempio deve violare la regola, altrimenti non dimostra nulla"
print("Confermato: la feature e' inammissibile.")

## Passo 5 - Split temporale contro split casuale

Una serie di temperatura oraria e' fortemente **autocorrelata**: l'ora `t` e
l'ora `t+1` si assomigliano molto di piu' di due ore scelte a caso a distanza
di mesi. Se lo split fra train e test viene fatto mescolando le righe a caso,
e' quasi certo che per ogni riga di test ci sia nel train un'ora
immediatamente adiacente: il modello non generalizza, ricorda un vicino, e la
metrica di test appare migliore di quanto sarebbe su dati davvero nuovi.

Lo split corretto (regola 4 del documento 02) taglia nel tempo: tutto cio'
che e' train viene prima, tutto cio' che e' test viene dopo. Il test e'
sempre il futuro rispetto al train, mai un campione a caso nel mezzo.

In [ ]:
ds = ds.sort_values("valid_time_utc").reset_index(drop=True)

# CORRETTO: taglio nel tempo. Il test e' il futuro rispetto al train.
taglio = ds["valid_time_utc"].quantile(0.7)
ds["split"] = np.where(ds["valid_time_utc"] <= taglio, "train", "test")
print("Split temporale:")
print(f"  train fino a {taglio}  -> {(ds['split']=='train').sum()} righe")
print(f"  test  dopo       -> {(ds['split']=='test').sum()} righe")

# SBAGLIATO, per confronto: mescolare le righe.
rng = np.random.default_rng(42)
split_casuale = rng.permutation(np.where(np.arange(len(ds)) < len(ds) * 0.7, "train", "test"))
sovrapposizione = (
    pd.Series(ds["valid_time_utc"][split_casuale == "test"]).min()
    < pd.Series(ds["valid_time_utc"][split_casuale == "train"]).max()
)
print(f"\nCon lo split casuale il test contiene istanti anteriori al train: {sovrapposizione}")
print("Su serie autocorrelate questo gonfia le metriche: ore adiacenti si somigliano,")
print("quindi il modello ritrova nel test cio' che ha gia' visto nel train.")

In [ ]:
colonne = ["station_id", "run_time_utc", "publication_time_utc", "lead_hours",
           "valid_time_utc", "t2m_c_forecast", "t2m_c_osservato", "errore", "split"]
ds[colonne].to_parquet(common.data_path("04_dataset.parquet"), index=False)
print("Scritto", common.data_path("04_dataset.parquet"))
print("Lo split e' una COLONNA, non tre file: nel notebook 05 il test resta")
print("visibile e va lasciato intatto fino alla valutazione finale.")

## Il limite di quello che hai fatto

- **Il join reale non ha prodotto righe.** Le osservazioni del notebook 02
  (2024) e la previsione del notebook 03 (scaricata il giorno stesso
  dell'esecuzione) non condividono nessun istante: e' la conseguenza diretta
  di aver costruito i due input in momenti diversi del calendario, non un
  difetto del metodo di join. Tutto cio' che segue il Passo 1 usa quindi un
  **dataset sintetico**, dichiarato come tale, generato con seed fisso: non e'
  un'osservazione reale e non va confuso con un risultato di verifica.
- **Il dataset (reale o sintetico) copre un periodo ridotto.** Nella
  simulazione, 30 giorni: il taglio temporale separa ore e settimane, non
  stagioni. Il vero split del progetto Nimbus separa anni interi (train
  2021-2024, validation 2025, test 2026, per blocchi di stagione: documento
  `docs/02-historical-forecast-observation-protocol.md`).
- **Manca la validation.** Il documento 02 prescrive tre insiemi (train,
  validation, test); qui ce ne sono solo due, perche' con cosi' poche righe
  un terzo taglio sarebbe illusorio: pochissimi punti per stimare la
  generalizzazione.
- **Il leakage mostrato e' il piu' evidente possibile**, una feature che
  copia apertamente un'osservazione futura. Ne esistono di piu' subdoli e
  piu' difficili da notare: per esempio una climatologia (una media storica
  per stazione e giorno dell'anno) calcolata sull'intero periodo disponibile,
  **incluso il test** — la regola 6 del documento 02 lo vieta esplicitamente
  per questo motivo, anche per aggregati che sembrano innocui perche' non
  sono previsioni dirette del singolo istante.